# C3PA Explorer -- Colab Snapshot Build

Builds `explorer.db` (the C3PA Explorer SQLite snapshot) on Google Colab by
running the **unchanged** deterministic pipeline (`backend.ingest` ->
`extract.py` + alignment) against a shallow clone of the official C3PA
dataset, then embeds a `manifest` table (`schema_version`, `dataset_commit`,
`code_rev`, `counts`, `db_sha256`, `built_at`) and downloads the single file
for the browser edition (sql.js file-picker -> in-memory database).

**Python >= 3.10 required** (the vendored backend uses PEP 604 `str | None`
annotations). Colab's default runtime (3.11 / 3.12) is fine; the notebook
asserts this in cell 2.

**How to rebuild:** run every cell in order, then pick `explorer.db` in the
browser edition. The notebook is regenerated from this checkout by
`scripts/build_colab_notebook.py` (vendored `%%writefile` cells -> no drift);
the dataset is shallow-cloned at HEAD, so `dataset_commit` (recorded in the
manifest) pins data provenance.

**Dataset layout note:** `git clone --depth 1` of
`MaazBinMusa/C3PA_Dataset.git` provides `Crawl/db.csv` + `Crawl/ws.csv` as
flat regular files (no submodule). The dataset's own README claims `Crawl/`
has `DB/`/`WS/` subfolders -- that is stale; **do not "fix"**
`load_crawl_meta`'s `Crawl/*{db,ws}*.csv` glob to match it.

**Build-time dependency:** only `beautifulsoup4==4.15.0` (pinned; it also
pulls `soupsieve` + `typing-extensions`). Extraction uses stdlib
`html.parser` only -- no lxml/html5lib, spacy, or fastapi.

**Expected full-corpus counts** (derived live from the built DB, not
hardcoded): 400 documents / 121,287 units / 84,985 sentences / 36,302
fragments / 45,121 annotations.


In [ ]:
# 1. Dataset: shallow clone (flat Crawl layout) + provenance
import os
import subprocess
import sys

assert sys.version_info >= (3, 10), "vendored backend requires Python >= 3.10 (PEP 604 `str | None` annotations)"

os.chdir("/content")

DATASET = "C3PA_Dataset"
DATASET_URL = "https://github.com/MaazBinMusa/C3PA_Dataset.git"

# An interrupted clone leaves a dir without Htmls/ -- git clone refuses to
# write into a non-empty dir, so in that case DELETE C3PA_Dataset and re-run.
if not os.path.isdir(os.path.join(DATASET, "Htmls")):
    !git clone --depth 1 {DATASET_URL} {DATASET}

# Integrity asserts. ingest's `ensure_dataset` ignores git exit codes, so do
# not trust it -- verify the artifacts the pipeline actually needs right here.
for req in (
    os.path.join(DATASET, "Htmls"),
    os.path.join(DATASET, "Crawl", "db.csv"),
    os.path.join(DATASET, "Crawl", "ws.csv"),
):
    assert os.path.exists(req), f"missing dataset artifact: {req}"

# Layout note: --depth 1 provides Crawl/db.csv + Crawl/ws.csv as flat regular
# files (no submodule). The dataset README's "Crawl has DB/WS subfolders" is
# stale -- do NOT 'fix' load_crawl_meta's glob to match it.

dataset_commit = subprocess.check_output(
    ["git", "-C", DATASET, "rev-parse", "HEAD"], text=True).strip()
print("dataset_commit:", dataset_commit)
print("dataset top-level:", sorted(os.listdir(DATASET)))


In [ ]:
# 2. Prepare /content/backend for the vendored files (belt-and-braces)
import os
os.makedirs("/content/backend", exist_ok=True)
print("backend/ ready:", os.path.isdir("/content/backend"))


In [ ]:
%%writefile backend/__init__.py


In [ ]:
%%writefile backend/db.py
"""SQLite schema for the C3PA Explorer database.

Two layers, kept fully separable and auditable:

SOURCE (verbatim from the C3PA dataset repo)
  documents    - 400 policies (Htmls/ + Crawl/ metadata)
  annotations  - 45,121 annotator spans (Annotations/, verbatim Text/Label)

DERIVED (deterministic re-extraction + annotation alignment, layered on top)
  text_units   - every extracted text unit, typed:
                   unit_kind = 'sentence' | 'fragment'
                   NOTE: "sentence" means *a complete thought or idea*, NOT the
                   literal grammarian's sentence. Terminal punctuation is a
                   proxy, not a requirement: authors omit trailing periods
                   before URLs/emails and in contact paragraphs, and one
                   thought is often stretched across a bulleted <li> list
                   (which the extractor joins into the text flow). See the
                   comment block in backend/extract.py:classify_unit for the
                   full rationale.
                   fragment_type = phone | email | url | nav | copyright | heading
                                   | list_item | lead_in | short | other
                   label_category applies to SENTENCES ONLY. Values:
                     'single_label' | 'multi_label' | 'unlabeled'.
                   'unlabeled' is MEASUREMENT-RELATIVE: for this annotation
                   version (6 annotators x budget) the sentence produced no
                   annotation signal in the annotators' net. It is NOT evidence
                   that no C3PA label applies and must not be treated as a
                   verified negative (positive-unlabeled caution) -- the net is
                   a function of resolution, not of label applicability.
                   Fragments are NOT categorically un-labelable -- they carry a
                   NULL label-state. By design we scope labels to sentences for
                   the sentence/label-pairs goal (a label on a non-sentence is
                   noise for that purpose), so fragments carry the distinct
                   value 'not-eligible' with no unit_labels rows; the alignment
                   table still records every annotation that touched them, so
                   their labels could be surfaced under other conditions.
  unit_labels      - M2M unit <-> verbatim C3PA labels (sentence units only)
  alignment        - provenance: which annotation supports which unit & how
                     (annotation_in_sentence | sentence_in_annotation_paragraph | partial_overlap)
"""

SCHEMA = """
CREATE TABLE IF NOT EXISTS documents (
    doc_id      TEXT PRIMARY KEY,          -- e.g. 'DB_1'
    subset      TEXT NOT NULL,             -- 'DB' | 'WS'
    num         INTEGER NOT NULL,
    title       TEXT,                      -- <title> tag of the crawled page
    link        TEXT,
    is_homepage TEXT,
    textmatch_p TEXT,
    textmatch_s TEXT,
    textmatch_pp TEXT,
    link_match  TEXT,
    html_path   TEXT NOT NULL,             -- provenance: C3PA Htmls/{subset}/{num}.html
    crawl_path  TEXT,
    UNIQUE (subset, num)
);

CREATE TABLE IF NOT EXISTS annotations (
    annotation_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    doc_id           TEXT NOT NULL REFERENCES documents(doc_id),
    ranumb           TEXT NOT NULL,        -- annotator: ra1..ra6
    text             TEXT NOT NULL,        -- verbatim annotation span
    label            TEXT NOT NULL,        -- verbatim C3PA label
    status           TEXT NOT NULL,        -- 'aligned' | 'unmatched' | 'ambiguous'
    matched_units    TEXT,                 -- populated when status = 'ambiguous'
    src_row          INTEGER NOT NULL,     -- original row index inside the source CSV
    source_csv       TEXT NOT NULL,        -- provenance: C3PA Annotations/{subset}/{num}.csv
    UNIQUE (doc_id, src_row)
);
CREATE INDEX IF NOT EXISTS idx_annotations_doc   ON annotations(doc_id);
CREATE INDEX IF NOT EXISTS idx_annotations_label ON annotations(label);
CREATE INDEX IF NOT EXISTS idx_annotations_status ON annotations(status);

CREATE TABLE IF NOT EXISTS text_units (
    unit_id   TEXT PRIMARY KEY,            -- e.g. 'DB_1_U12'
    doc_id    TEXT NOT NULL REFERENCES documents(doc_id),
    position  INTEGER NOT NULL,            -- ordinal within the document
    block_seq INTEGER NOT NULL,            -- containing block index (for rendering)
    block_kind TEXT NOT NULL,              -- 'prose' | 'heading' | 'list'
    unit_text TEXT NOT NULL,
    unit_kind TEXT NOT NULL,               -- 'sentence' | 'fragment'
    fragment_type TEXT,                    -- for fragments
    label_category TEXT NOT NULL,          -- 'single_label' | 'multi_label' | 'unlabeled'
    annotator_count INTEGER NOT NULL DEFAULT 0,
    source_annotation_count INTEGER NOT NULL DEFAULT 0,
    alignment_types TEXT,                  -- ';'-joined alignment strategies
    UNIQUE (doc_id, position)
);
CREATE INDEX IF NOT EXISTS idx_units_doc   ON text_units(doc_id);
CREATE INDEX IF NOT EXISTS idx_units_kind  ON text_units(unit_kind);

CREATE TABLE IF NOT EXISTS unit_labels (
    unit_id TEXT NOT NULL REFERENCES text_units(unit_id),
    label   TEXT NOT NULL,
    PRIMARY KEY (unit_id, label)
);
CREATE INDEX IF NOT EXISTS idx_unit_labels_label ON unit_labels(label);

CREATE TABLE IF NOT EXISTS alignment (
    unit_id        TEXT NOT NULL REFERENCES text_units(unit_id),
    annotation_id  INTEGER NOT NULL REFERENCES annotations(annotation_id),
    alignment_type TEXT NOT NULL,
    PRIMARY KEY (unit_id, annotation_id)
);
CREATE INDEX IF NOT EXISTS idx_alignment_ann ON alignment(annotation_id);
"""

UNIT_FIELDS = [
    "id", "doc_id", "group", "text", "label", "label_name",
    "subset", "position", "unit_kind", "fragment_type", "label_category",
    "verbatim_labels", "annotator_count", "annotators",
    "source_annotation_count", "alignment_types",
]

# ---- connection helpers -------------------------------------------------------

DB_PATH = None


def connect():
    import sqlite3

    path = DB_PATH or "data/explorer.db"
    conn = sqlite3.connect(path, check_same_thread=False)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA busy_timeout = 5000")
    return conn


def init_db(conn):
    conn.executescript(SCHEMA)
    conn.commit()


# ---- shared unit filter builder -----------------------------------------------

def support_tier_cond(mode: str, tier: int) -> str:
    """SQL predicate for a unit's support tier.

    mk = max distinct-annotator support across the unit's label pairs, n = the
    document's annotator pool. tiers: 3 unanimous (mk==n), 2 majority
    (mk>=ceil(n/2)), 1 minimal (mk>=1). mode 'at_least' matches 'this tier or
    greater'; 'exact' matches only units whose best support is exactly the tier.
    """
    ceil = "CAST((pl.n + 1) / 2 AS INTEGER)"
    if mode == "exact":
        return {
            3: "x.mk = pl.n",
            2: f"x.mk >= {ceil} AND x.mk < pl.n",
            1: f"x.mk >= 1 AND x.mk < {ceil}",
        }.get(tier, "x.mk >= 1")
    return {
        3: "x.mk = pl.n",
        2: f"x.mk >= {ceil}",
        1: "x.mk >= 1",
    }.get(tier, "x.mk >= 1")


def unit_support(conn, threshold: int = 1) -> dict:
    """Per-(unit,label) annotator support at an evidence threshold.

    k = distinct annotators with at least `threshold` aligned annotations for
    that (unit,label) pair; pool = the document's distinct annotator pool.
    stars: 3 unanimous (k == pool), 2 majority (k >= ceil(pool/2)), 1 minority
    (ceil = 50/50 rounds up; a single-annotator doc is always unanimous).
    Returns {unit_id: [{'label','k','pool','stars'}, ...]} for pairs with k >= 1.
    """
    import math

    pool = {r["doc_id"]: r["n"] for r in conn.execute(
        "SELECT doc_id, COUNT(DISTINCT ranumb) n FROM annotations GROUP BY doc_id")}
    votes = {}
    for r in conn.execute(
            """SELECT g.unit_id, u.doc_id, sl.label, a.ranumb, COUNT(*) c
               FROM unit_labels sl
               JOIN alignment g ON g.unit_id = sl.unit_id
               JOIN text_units u ON u.unit_id = g.unit_id
               JOIN annotations a ON a.annotation_id = g.annotation_id AND a.label = sl.label
               GROUP BY g.unit_id, u.doc_id, sl.label, a.ranumb"""):
        if r["c"] >= threshold:
            v = votes.setdefault((r["unit_id"], r["label"]), {"doc": r["doc_id"], "n": 0})
            v["n"] += 1
    out = {}
    for (unit_id, label), v in votes.items():
        p = pool.get(v["doc"], 0)
        k = v["n"]
        stars = 3 if k == p else 2 if k >= math.ceil(p / 2) else 1
        out.setdefault(unit_id, []).append({"label": label, "k": k, "pool": p, "stars": stars})
    return out


def build_unit_filter(params: dict) -> tuple[str, list]:
    """Builds a WHERE clause for unit queries from filter params.

    Supported keys:
      subset        - 'DB' | 'WS'
      doc_id        - exact document id
      category      - label category: 'single_label' | 'multi_label' | 'unlabeled'
                      (sentences only; fragments are 'not-eligible')
      unit_kind     - 'sentence' | 'fragment'
      fragment_type - fragment sub-type (when unit_kind == 'fragment')
      labels        - list of verbatim labels (matches units carrying ANY of them)
      exclude_labels- list of verbatim labels (EXCLUDES units carrying ANY of them)
      min_annotators- annotator_count >= N
      high_confidence - bool; single_label AND annotator_count >= 2
      q             - substring search over unit_text
      sample_view   - list of sample-family chips, OR'd together:
                      'single' | 'multi' | 'null' | 'fragments'
      evidence_threshold - int (default 1); for the 'single'/'multi' families a
                      label pair is supported only when some annotator produced
                      >= this many aligned annotations for it, so raising it
                      prunes underdefined sentences from the defined slice.
      min_support   - int 1..3 (default 1 = Minimal); only 'single'/'multi'
                      units with at least one label pair meeting the support
                      tier remain: 3 unanimous (all annotators), 2 majority
                      (>= ceil(pool/2) annotators), 1 minimal (>= 1).
      support_mode  - 'at_least' (default; this tier or greater) | 'exact'
                      (only records with exactly the tier).
    """
    where = []
    args = []

    if params.get("subset"):
        where.append("d.subset = ?")
        args.append(params["subset"])
    if params.get("doc_id"):
        where.append("u.doc_id = ?")
        args.append(params["doc_id"])
    if params.get("category"):
        where.append("u.label_category = ?")
        args.append(params["category"])
    if params.get("unit_kind"):
        where.append("u.unit_kind = ?")
        args.append(params["unit_kind"])
    if params.get("fragment_type"):
        where.append("u.fragment_type = ?")
        args.append(params["fragment_type"])
    if params.get("min_annotators"):
        where.append("u.annotator_count >= ?")
        args.append(int(params["min_annotators"]))
    if params.get("high_confidence"):
        where.append("u.label_category = 'single_label' AND u.annotator_count >= 2")
    if params.get("q"):
        where.append("u.unit_text LIKE ? ESCAPE '\\'")
        q = params["q"].replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")
        args.append(f"%{q}%")
    if params.get("labels"):
        placeholders = ",".join("?" for _ in params["labels"])
        where.append(
            "u.unit_id IN (SELECT unit_id FROM unit_labels WHERE label IN (%s))"
            % placeholders
        )
        args.extend(params["labels"])
    if params.get("exclude_labels"):
        placeholders = ",".join("?" for _ in params["exclude_labels"])
        where.append(
            "u.unit_id NOT IN (SELECT unit_id FROM unit_labels WHERE label IN (%s))"
            % placeholders
        )
        args.extend(params["exclude_labels"])

    t = int(params.get("evidence_threshold") or 1)
    ms = int(params.get("min_support") or 1)
    if ms not in (1, 2, 3):
        ms = 1
    mode = params.get("support_mode") or "at_least"
    tier_cond = support_tier_cond(mode, ms)
    supported = (
        "u.unit_id IN (SELECT x.unit_id FROM ("
        "SELECT p.unit_id, p.doc_id, MAX(p.k) mk FROM ("
        "SELECT q.unit_id, q.doc_id, q.label, COUNT(*) k FROM ("
        "SELECT g.unit_id, u.doc_id, sl.label, a.ranumb FROM unit_labels sl "
        "JOIN alignment g ON g.unit_id = sl.unit_id "
        "JOIN text_units u ON u.unit_id = g.unit_id "
        "JOIN annotations a ON a.annotation_id = g.annotation_id AND a.label = sl.label "
        "GROUP BY g.unit_id, u.doc_id, sl.label, a.ranumb HAVING COUNT(*) >= ?) q "
        "GROUP BY q.unit_id, q.doc_id, q.label) p "
        "GROUP BY p.unit_id, p.doc_id) x "
        "JOIN (SELECT doc_id, COUNT(DISTINCT ranumb) n FROM annotations GROUP BY doc_id) pl "
        "ON pl.doc_id = x.doc_id "
        f"WHERE {tier_cond})"
    )
    if params.get("sample_view"):
        views = [v for v in params["sample_view"] if v in ("single", "multi", "null", "fragments")]
        if views:
            # A family that can never satisfy an active label/support filter
            # yields nothing: null/fragments carry no labels and no support.
            quality = bool(
                (ms > 1 or mode == "exact") or t > 1
                or params.get("labels") or params.get("exclude_labels")
            )
            conds = {
                "single": f"(u.unit_kind = 'sentence' AND u.label_category = 'single_label' AND {supported})",
                "multi": f"(u.unit_kind = 'sentence' AND u.label_category = 'multi_label' AND {supported})",
                "null": "(u.unit_kind = 'sentence' AND u.label_category = 'unlabeled')" if not quality else "(0)",
                "fragments": "(u.unit_kind = 'fragment')" if not quality else "(0)",
            }
            where.append("(" + " OR ".join(conds[v] for v in views) + ")")
            args.extend([t] * sum(1 for v in views if v in ("single", "multi")))
    elif ms > 1 or t > 1 or mode == "exact" or params.get("labels") or params.get("exclude_labels"):
        # "All" (no sample_view) with a label or support filter active: only
        # samples that can satisfy the requirement remain -- the defined
        # (single_label / multi_label) families passing the support gate.
        # Unlabeled sentences and fragments carry no labels/support, so they
        # can never meet the requirement and are excluded.
        where.append(
            f"(u.label_category IN ('single_label', 'multi_label') AND {supported})"
        )
        args.extend([t])

    if where:
        return " WHERE " + " AND ".join(where), args
    return "", args


UNIT_SELECT = """
    SELECT u.unit_id, u.doc_id, d.subset, u.position, u.block_kind, u.unit_text,
           u.unit_kind, u.fragment_type, u.label_category,
           u.annotator_count, u.source_annotation_count, u.alignment_types,
           (SELECT group_concat(sl.label, ';') FROM unit_labels sl
             WHERE sl.unit_id = u.unit_id ORDER BY sl.label) AS verbatim_labels,
           (SELECT group_concat(ranumb, ';') FROM
             (SELECT DISTINCT a.ranumb FROM alignment g
               JOIN annotations a ON a.annotation_id = g.annotation_id
               WHERE g.unit_id = u.unit_id ORDER BY a.ranumb)) AS annotators
    FROM text_units u
    JOIN documents d ON d.doc_id = u.doc_id
"""

In [ ]:
%%writefile backend/extract.py
"""Deterministic HTML -> text-unit extraction for the C3PA Explorer.

Replaces the parser's line-based spaCy sentencizer with a fully rule-based,
auditable pipeline:

    HTML
      -> structural pass  (drop chrome containers: nav/header/footer/menu/...)
      -> leaf text blocks (paragraphs, headings, list items)
      -> rule-based sentence splitter (abbreviation/initial/decimal/URL guards)
      -> unit classifier  (sentence | fragment + typed fragment sub-class)

No trained models anywhere - every decision is a deterministic rule.
"""

import csv
import os
import re
import unicodedata
from bs4 import BeautifulSoup

# ---------------------------------------------------------------------------
# 1. Structural chrome classification
# ---------------------------------------------------------------------------

CHROME_TAGS = {
    "nav", "header", "footer", "aside", "button", "select", "input",
    "textarea", "script", "style", "noscript", "svg", "canvas", "iframe",
    "template", "dialog", "audio", "video",
}

# class/id tokens (compared against normalized "foo bar" form, so hyphens
# and underscores collapse to spaces). Word-sequence match with a guard so a
# token preceded by "no"/"non"/"article" is NOT treated as chrome (e.g. Best
# Buy's "no-header-paragraph" and "article header__small" content classes).
CHROME_TOKENS = [
    "navbar", "topbar", "top bar", "breadcrumb", "cookie", "consent", "modal",
    "popup", "drawer", "subscribe", "newsletter", "signup", "sign in", "signin",
    "login", "logout", "sidebar", "pagination", "advertisement", "ad banner",
    "announcement", "promo", "toast", "toolbar", "chat", "drift", "messenger",
    "back to top", "secondary menu", "top navigation", "mobile nav",
    "site header", "site footer", "main header", "main footer", "page header",
    "page footer", "top header", "top footer", "footer", "header", "menu",
    "nav", "main menu", "primary menu", "menu container", "et info",
    "tb footer", "tb header", "acsb", "masthead",
]
_CHROME_GUARD = {"no", "non", "article"}

# Bare generic words that are only treated as chrome when the class/id is
# SHORT. Long compound classes like "closed-mobile-header" or
# "no-header-paragraph" are often page wrappers or content, not chrome --
# matching the bare word there would drop the whole document (e.g. DB_8).
# "topbar" joins the list because full-page wrappers like
# "body_wrapper header_topbar" (DB_21) contain the ENTIRE document; a real top
# bar is a standalone "topbar"/"site topbar" (<= 2 words), which still matches.
_SHORT_ONLY = {"header", "footer", "nav", "menu", "sidebar", "promo", "topbar"}
# a matched generic word followed by a layout negation ("sidebar-none",
# "no-sidebar") describes the page layout, not a chrome region
_NEGATION = {"none", "no", "without", "absent", "closed", "hidden", "off"}
# a short-only chrome token followed by one of these is a page wrapper /
# content region, NOT chrome: "nav-content", "header-wrapper",
# "footer-container" hold the actual policy text (WS_38, WS_94). The explicit
# multiword chrome tokens ("menu container", "site header", ...) are still
# checked separately and catch genuine chrome containers.
_CONTENT_WRAP = {"content", "wrapper", "wrap", "container", "holder", "main",
                 "body", "area", "section", "region"}

BLOCK_CONTENT_TAGS = {
    "p", "h1", "h2", "h3", "h4", "h5", "h6", "li", "td", "th", "dt", "dd",
    "blockquote", "figcaption", "pre", "caption",
}

# tags that make a generic container a "container" (recurse) rather than a leaf
BLOCK_STRUCT_TAGS = {
    "p", "div", "section", "article", "h1", "h2", "h3", "h4", "h5", "h6",
    "ul", "ol", "li", "table", "thead", "tbody", "tfoot", "tr", "td", "th",
    "blockquote", "pre", "figure", "figcaption", "header", "footer", "nav",
    "aside", "main", "form", "fieldset", "address", "hr", "dl", "dt", "dd",
}

# inline chrome to strip from inside content blocks
INLINE_STRIP_TAGS = {"button", "input", "select", "textarea", "img", "svg",
                     "iframe", "canvas", "video", "audio", "script", "style"}

# inline text elements whose content is captured by their parent block
INLINE_SKIP_TAGS = {"a", "span", "strong", "em", "i", "b", "u", "font", "small",
                    "sub", "sup", "label", "abbr", "code", "time", "mark", "ins",
                    "del", "cite", "q", "br", "wbr", "bdo", "big"}

_WS = re.compile(r"\s+")


def _normalize_attrs(node) -> str:
    classes = " ".join(node.get("class") or [])
    node_id = node.get("id") or ""
    return _WS.sub(" ", f"{classes} {node_id}".replace("-", " ").replace("_", " ")).lower()


def is_chrome_node(node) -> bool:
    if not getattr(node, "name", None):
        return False
    if node.name in CHROME_TAGS:
        return True
    hay = _WS.sub(" ", _normalize_attrs(node)).split()
    for tok in CHROME_TOKENS:
        words = tok.split()
        for i in range(len(hay) - len(words) + 1):
            if hay[i:i + len(words)] == words:
                if i == 0 or hay[i - 1] not in _CHROME_GUARD:
                    if tok in _SHORT_ONLY:
                        if len(hay) > 2:
                            continue
                        if i + len(words) < len(hay):
                            nxt = hay[i + len(words)]
                            if nxt in _NEGATION or nxt in _CONTENT_WRAP:
                                continue
                    return True
    return False


def extract_title(html: str) -> str:
    """The document's <title> tag (the browser-tab title of the crawled page)."""
    soup = BeautifulSoup(html, "html.parser")
    t = soup.find("title")
    if t is None:
        return ""
    return _WS.sub(" ", t.get_text(" ", strip=True)).strip()


def extract_blocks(html: str) -> list[dict]:
    """Walks the DOM and emits leaf text blocks in document order.

    Each block is {kind: 'prose'|'heading'|'list', text: <collapsed text>}.
    """
    soup = BeautifulSoup(html, "html.parser")
    for tag in ("script", "style", "noscript", "head", "svg", "title"):
        for el in soup.find_all(tag):
            el.decompose()
    for el in soup.find_all(INLINE_STRIP_TAGS):
        el.decompose()

    blocks: list[dict] = []

    def emit(name, t):
        t = _WS.sub(" ", t).strip()
        if t and any(c.isalnum() for c in t):
            kind = ("heading" if name in ("h1", "h2", "h3", "h4", "h5", "h6")
                    else "list" if name == "li" else "prose")
            blocks.append({"kind": kind, "text": t})

    def process(node, depth=0):
        if depth > 60:
            return
        if getattr(node, "name", None) is None:
            return
        if is_chrome_node(node):
            return
        name = node.name
        if name in INLINE_SKIP_TAGS:
            if not node.find(list(BLOCK_STRUCT_TAGS)):
                return
        if name in BLOCK_CONTENT_TAGS:
            emit(name, node.get_text(" ", strip=True))
            return
        if name in ("ul", "ol"):
            # Lists join the text flow: <li> is presentation, exactly like
            # <p>/<div>/<a>/<br>. The item text already carries the ";" and
            # "and" separators, so joining the items yields the sentence a
            # reader actually sees. (Design note: "sentence" here means a
            # *complete thought or idea*; a policy author frequently stretches
            # one thought across a bulleted list, e.g. "you may have the
            # following additional rights: the right to access; the right to
            # rectification; ...". We do not hoard the literal <li> boundary.)
            items = []
            for li in node.find_all("li", recursive=True):
                t = _WS.sub(" ", li.get_text(" ", strip=True)).strip()
                if t:
                    items.append(t)
            if items:
                blocks.append({"kind": "list", "text": " ".join(items)})
            return
        if name in ("table", "tbody", "thead", "tfoot", "tr"):
            cells = node.find_all(["td", "th"], recursive=False)
            if cells:
                for cell in cells:
                    process(cell, depth + 1)
                return
            # no direct cell children (e.g. table > tbody > tr > td): fall
            # through to generic container recursion instead of dropping the
            # whole table (DB_21 stores all its policy text in such tables)
        # generic container: recurse only if it has block-structure children;
        # otherwise it is a leaf text container (e.g. div > "direct text")
        block_children = [c for c in node.children
                          if getattr(c, "name", None) and c.name in BLOCK_STRUCT_TAGS]
        if not block_children:
            emit(name, node.get_text(" ", strip=True))
            return
        # mixed content: leading/inline text before block children (e.g.
        # "<div>intro sentence...<p>rest</p></div>")
        inline_copy = BeautifulSoup(str(node), "html.parser")
        root = inline_copy.find()
        if root is not None:
            for b in root.find_all(BLOCK_STRUCT_TAGS):
                b.decompose()
            lead = _WS.sub(" ", root.get_text(" ", strip=True)).strip()
            if lead:
                emit(name, lead)
        for child in node.children:
            process(child, depth + 1)

    for child in soup.body.children if soup.body else []:
        process(child)

    return _merge_leadin_lists(blocks)


def _merge_leadin_lists(blocks: list[dict]) -> list[dict]:
    """A prose block ending in ':' introduces whatever follows, so a lead-in
    immediately followed by a list is really ONE complete thought:

        "...you may have the following additional rights:"  +  <ul>items</ul>
            -> "...you may have the following additional rights: the right to
                access; ... and the right to complain to a supervisory authority"

    This mirrors how the pipeline already treats other tag boundaries as
    presentation; the lead-in's colon is the grammatical join, not the <ul> tag.
    """
    merged: list[dict] = []
    i = 0
    while i < len(blocks):
        b = blocks[i]
        nxt = blocks[i + 1] if i + 1 < len(blocks) else None
        if (nxt and nxt["kind"] == "list" and b["kind"] == "prose"
                and b["text"].rstrip().endswith(":")):
            merged.append({"kind": "prose",
                           "text": b["text"].rstrip() + " " + nxt["text"]})
            i += 2
        else:
            merged.append(b)
            i += 1
    return merged


# ---------------------------------------------------------------------------
# 2. Deterministic sentence splitter
# ---------------------------------------------------------------------------

_ABBREVIATIONS = {
    "mr", "mrs", "ms", "miss", "dr", "prof", "rev", "capt", "col", "gen", "lt",
    "sgt", "gov", "sen", "rep", "dept", "est", "inc", "ltd", "corp", "co",
    "jr", "sr", "i.e", "e.g", "etc", "vs", "ph.d", "u.s", "u.k", "u.s.a",
    "no", "nos", "fig", "vol", "st", "mt", "ft", "sec", "min", "hrs", "approx",
    "esq", "univ", "calif", "ave", "blvd", "ct", "rd",
    "a.m", "p.m", "am", "pm", "jan", "feb", "mar", "apr", "jun", "jul", "aug",
    "sep", "sept", "oct", "nov", "dec", "mon", "tue", "wed", "thu", "fri", "sat", "sun",
}

# note: the value is only used as a set lookup on the lowercased token before '.'
_ABBREV_SET = _ABBREVIATIONS

_CLOSING = "”\"'）)]»"


_URL_EMAIL = re.compile(
    r"https?://[^\s,;()]+|www\.[^\s,;()]+|[\w.+-]+@[\w.-]+\.[a-z]{2,}", re.I)


def _mask(text: str):
    """Replaces URL/email substrings with inert placeholders so their internal
    periods never trigger sentence boundaries. Returns (masked, originals).

    A trailing sentence-final period after a URL is *not* part of the URL: it
    is re-emitted after the placeholder so it stays a sentence boundary.
    """
    parts = []

    def repl(m):
        raw = m.group(0)
        tail = ""
        if raw.endswith((".", "!", "?")):
            tail = raw[-1]
            raw = raw[:-1]
        parts.append(raw)
        return f" \u0001{len(parts) - 1}\u0001 {tail}"

    masked = _URL_EMAIL.sub(repl, text)
    return masked, parts


def split_sentences(text: str) -> list[str]:
    """Splits paragraph text into sentence candidates using explicit rules."""
    masked, parts = _mask(text)
    units: list[str] = []
    start = 0
    n = len(masked)
    i = 0
    while i < n:
        ch = masked[i]
        if ch in ".!?" and _is_boundary(masked, i):
            j = i + 1
            while j < n and masked[j] in _CLOSING:
                j += 1
            units.append(masked[start:j].strip())
            start = j
            i = j
        else:
            i += 1
    tail = masked[start:].strip()
    if tail:
        units.append(tail)
    units = [u for u in units if u]
    # restore protected substrings
    for k, u in enumerate(units):
        units[k] = re.sub(r"\u0001(\d+)\u0001",
                          lambda m: parts[int(m.group(1))], u)
    return units


def _next_nonspace(text: str, i: int):
    j = i + 1
    while j < len(text) and text[j] in (" ", _CLOSING):
        j += 1
    return j


def _is_boundary(text: str, i: int) -> bool:
    ch = text[i]

    # '!' and '?' are always sentence-final (guard against !?/?! sequences)
    if ch in "!?":
        if i + 1 < len(text) and text[i + 1] in "!?":
            return False
        return True

    # ellipsis / repeated periods
    if (i + 1 < len(text) and text[i + 1] == ".") or (i > 0 and text[i - 1] == "."):
        return False

    # scan backwards over closing quotes/parens, whitespace and masked
    # placeholders to find the "real" token before the period
    j = i - 1
    while j >= 0:
        c = text[j]
        if c in " \t\n" or c in _CLOSING:
            j -= 1
        elif c == "\u0001":          # end of a masked URL/email placeholder
            k = j
            while k >= 0 and text[k] != "\u0001":
                k -= 1
            j = k - 1
        else:
            break
    if j < 0 or not text[j].isalnum():
        return False

    # abbreviation / initial / URL checks on the token ending at j
    prev_start = j
    while prev_start > 0 and text[prev_start - 1].isalnum():
        prev_start -= 1
    prev_token = text[prev_start:j + 1].lower()
    if prev_token in _ABBREV_SET:
        return False
    if len(prev_token) == 1 and prev_token.isalpha():
        return False
    if "/" in text[prev_start:j + 1]:
        return False

    # next character after optional closing quotes
    nxt = _next_nonspace(text, i)
    if nxt >= len(text):
        return True
    nxt_ch = text[nxt]

    if nxt_ch.isdigit():
        return False
    if nxt_ch.islower():
        return False
    # next is uppercase -> sentence boundary
    return True


# ---------------------------------------------------------------------------
# 3. Unit classifier (sentence vs fragment + sub-types)
# ---------------------------------------------------------------------------

# deterministic verb probe: privacy-policy vocabulary + auxiliaries/modals
VERB_PROBE = {
    # auxiliaries / modals
    "is", "are", "was", "were", "be", "been", "being", "am", "have", "has",
    "had", "do", "does", "did", "may", "might", "can", "could", "will", "would",
    "shall", "should", "must", "need", "ought",
    # lexical verbs common in privacy policies
    "use", "uses", "used", "using", "collect", "collects", "collected", "collecting",
    "share", "shares", "shared", "sharing", "disclose", "discloses", "disclosed",
    "disclosing", "sell", "sells", "sold", "selling", "delete", "deletes",
    "deleted", "deleting", "store", "stores", "stored", "storing", "retain",
    "retains", "retained", "retaining", "process", "processes", "processed",
    "processing", "provide", "provides", "provided", "providing", "offer",
    "offers", "offered", "offering", "require", "requires", "required",
    "requiring", "allow", "allows", "allowed", "allowing", "permit", "permits",
    "permitted", "permit", "apply", "applies", "applied", "access", "accesses",
    "accessed", "transfer", "transfers", "transferred", "transferring",
    "receive", "receives", "received", "receiving", "obtain", "obtains",
    "obtained", "obtaining", "protect", "protects", "protected", "protecting",
    "safeguard", "safeguards", "safeguarded", "secure", "secures", "secured",
    "maintain", "maintains", "maintained", "maintaining", "update", "updates",
    "updated", "updating", "change", "changes", "changed", "changing", "inform",
    "informs", "informed", "informing", "notify", "notifies", "notified",
    "notifying", "contact", "contacts", "contacted", "contacting", "request",
    "requests", "requested", "requesting", "consent", "consents", "consented",
    "object", "objects", "objected", "restrict", "restricts", "restricted",
    "correct", "corrects", "corrected", "erase", "erases", "erased", "remove",
    "removes", "removed", "removing", "govern", "governs", "governed",
    "control", "controls", "controlled", "limit", "limits", "limited",
    "track", "tracks", "tracked", "log", "logs", "logged", "record", "records",
    "recorded", "publish", "publishes", "published", "post", "posts", "posted",
    "display", "displays", "displayed", "send", "sends", "sent", "sending",
    "deliver", "delivers", "delivered", "transmit", "transmits", "transmitted",
    "comply", "complies", "complied", "ensure", "ensures", "ensured",
    "describe", "describes", "described", "contain", "contains", "contained",
    "include", "includes", "included", "cover", "covers", "covered",
    "explain", "explains", "explained", "state", "states", "stated",
    "outline", "outlines", "outlined", "summarize", "summarizes", "summarized",
    "define", "defines", "defined", "means", "refers", "refer", "referred",
    "relate", "relates", "related", "belongs", "concern", "concerns",
    "concerned", "represent", "represents", "represented", "let", "lets",
    "make", "makes", "made", "take", "takes", "taken", "give", "gives", "given",
    "look", "looks", "follow", "follows", "followed", "continue", "continues",
    "continued", "reserve", "reserves", "reserved", "review", "reviews",
    "reviewed", "expect", "expects", "expected", "happen", "happens", "happened",
    "agree", "agrees", "agreed", "accept", "accepts", "accepted", "remain",
    "remains", "remained", "believe", "believes", "believed", "understand",
    "understands", "understood", "know", "knows", "known", "note", "notes",
    "noted", "assume", "assumes", "assumed", "assure", "assures", "assured",
}

# exact-match chrome labels (whole unit, lowercased)
CHROME_LABELS = {
    "login", "log in", "sign in", "sign up", "register", "contact us", "about us",
    "home", "menu", "search", "faq", "help", "get started", "free trial",
    "pricing", "privacy policy", "terms", "terms of use", "terms & conditions",
    "terms and conditions", "cookie policy", "sitemap", "language", "english",
    "español", "espanol", "more", "learn more", "read more", "accept", "agree",
    "decline", "ok", "okay", "close", "submit", "cancel", "back", "next",
    "continue", "shop", "cart", "account", "my account", "support", "download",
    "subscribe", "unsubscribe", "newsletter", "all rights reserved", "back to top",
}

_PHONE = re.compile(r"^\+?[\d][\d\s().\-]{5,}$")
_PHONE2 = re.compile(r"^\(?\d{3}\)?[\s.\-]?\d{3}[\s.\-]?\d{4}$")
_EMAIL = re.compile(r"^[\w.+-]+@[\w.-]+\.[a-z]{2,}$", re.I)
# scheme-less domains too ("VDX.tv", "example.com"), so a sentence-final bare
# domain is recognized as a URL-final complete thought ("Learn more at VDX.tv .")
_URL = re.compile(r"^(https?://|www\.|([a-z0-9-]+\.)+[a-z]{2,})(/\S*)?$", re.I)
_COPYRIGHT = re.compile(r"^(©|copyright\b).*(\d{4}|all rights reserved)", re.I)
_NUMBERED_HEADING = re.compile(r"^\d+(\.\d+)*[\.\:]?\s+\S")
_ROMAN_HEADING = re.compile(r"^[IVX]+\.?\s+\S")
_ALLCAPS_SHORT = re.compile(r"^[A-Z0-9\s\-/&.'’%]{2,}$")
_LIST_ITEM_END = re.compile(r";(\s+(and|or))?$")
# words that begin a *dependent* clause; without a following main clause the
# unit is NOT a complete thought (e.g. "When you use our Services" alone)
_SUBORDINATE_START = re.compile(
    r"^(when|if|although|because|unless|while|after|before|since|whereas|"
    r"whether|as|though|until|provided)\b", re.I)

# Structural finite-verb signals. VERB_PROBE is a curated wordlist with
# inherent gaps ("pledges", "conduct", "recommend", ...), and enumerating verbs
# forever is whack-a-mole. These two *structural* patterns catch the gap
# classes with high precision and no wordlist growth:
#   1) a modal/auxiliary (or its contraction): "we MAY need", "we CANNOT
#      guarantee", "you MUST opt out"
#   2) a subject pronoun directly before a verb: "WE recommend", "YOU waive",
#      "THEY collect"
_MODAL_AUX = re.compile(
    r"\b(am|is|are|was|were|have|has|had|do|does|did|may|might|must|can|could|"
    r"will|would|shall|should|need|ought|cannot|can't|won't|don't|doesn't|didn't|"
    r"isn't|aren't|wasn't|weren't|haven't|hasn't|hadn't|shouldn't|couldn't|"
    r"wouldn't|mustn't|needn't)\b", re.I)
_SUBJECT_VERB = re.compile(r"\b(we|you|they|i|it)\s+[a-z]+\b", re.I)
# 3) "please <verb>" -- a politeness-marker imperative ("Please read this
#    Policy carefully.", "please visit the site", "please contact us")
_PLEASE_IMPERATIVE = re.compile(r"\bplease\s+[a-z]+\b", re.I)
# 4) a word ending in -s/-ed directly before a determiner or adverb -- the
#    noun-subject finite-verb signature that the pronoun/aux rules cannot see:
#    "VDX.tv acts ethically", "The ad server checks the ...",
#    "Experian facilitated the ...", "Company has disclosed the ..."
_S_ED_DET_ADV = re.compile(
    r"\b(\w+(?:s|ed))\s+(the|this|these|those|our|your|their|its|his|her|a|an|\w+ly)\b",
    re.I)


def _has_finite_verb_signal(t: str) -> bool:
    """True when the text carries a structure-based sign of a finite verb.

    Known residual gap (accepted per design): bare present-tense verbs after a
    plural-noun subject ("Our Partners perform ...") and bare imperatives
    without "please" ("Click here to opt-out") are indistinguishable from
    nouns without a parser, so those remain fragments. This is the deliberate
    conservative direction -- see classify_unit.
    """
    return bool(_MODAL_AUX.search(t) or _SUBJECT_VERB.search(t)
                or _PLEASE_IMPERATIVE.search(t) or _S_ED_DET_ADV.search(t))


def _is_phone(t: str) -> bool:
    if sum(c.isdigit() for c in t) < 7:
        return False
    return bool(_PHONE.match(t) or _PHONE2.match(t))


# trailing whitespace + sentence punctuation + closing quotes/parens
_TRAILING_PUNCT = re.compile(r"[\s.!?;:\u201d\u201c\"'）)\]»]+$")


def _ends_in_url_email(t: str) -> bool:
    """True when the unit's final token is a URL or email.

    Policy editors terminate a URL-final sentence in (at least) two ways, and
    both must count as sentence-final:
        DB_1 convention: "...policy at https://example.com/terms"      (no period)
        DB_2 convention: "...policy at https://example.com/terms ."    (space, period)
    A trailing period would alter the resource locator, so both conventions
    omit the period *glued to the URL*; we strip trailing punctuation/space
    and test the last real token.
    """
    stripped = _TRAILING_PUNCT.sub("", t)
    toks = stripped.split()
    if not toks:
        return False
    return bool(_URL.match(toks[-1]) or _EMAIL.match(toks[-1]))


def classify_unit(text: str, block_kind: str) -> tuple[str, str | None]:
    """Returns (unit_kind, fragment_type).

    unit_kind: 'sentence' | 'fragment'
    fragment_type (fragments only): phone, email, url, nav, button, copyright,
        heading, list_item, lead_in, short, other

    WHAT "SENTENCE" MEANS HERE
    --------------------------
    "Sentence" is shorthand for *a complete thought or idea*. English sentences
    usually map onto that concept, which is why terminal punctuation + length +
    a verb probe are a good proxy. But three corpus realities mean period
    presence must NOT be a hard requirement:

      1. A sentence-final URL/email carries no glued-on period (a trailing '.'
         would alter the resource locator). Editors differ: some end the
         sentence with no period at all ("...at example.com/terms"), others
         add " ." with a space ("...at example.com/terms ."). _ends_in_url_email
         recognizes both conventions as sentence-final.
      2. Contact/opt-out paragraphs routinely drop sentence-final punctuation.
      3. One thought is often stretched across a bulleted <li> list.

    So sentencehood is decided by: substantive length + a finite-verb probe +
    "does not look like chrome / a heading / a lone subordinate clause" + a
    sentence-final URL/email as an independent complete-thought signal.
    This is deliberately conservative in the *fragment* direction: mis-typing a
    heading as a sentence pollutes the sentence view, whereas an over-split
    complete thought remains visible as a typed fragment.
    """
    t = text.strip()
    if not t:
        return ("fragment", "other")
    lower = t.lower()

    # whole-unit chrome patterns
    if _is_phone(t):
        return ("fragment", "phone")
    if _EMAIL.match(t):
        return ("fragment", "email")
    if _URL.match(t):
        return ("fragment", "url")
    if _COPYRIGHT.match(t):
        return ("fragment", "copyright")
    if lower in CHROME_LABELS:
        return ("fragment", "nav" if len(lower.split()) <= 3 else "button")

    words = t.split()
    wc = len(words)
    has_verb = any(w.lower() in VERB_PROBE for w in words)
    has_terminal = bool(re.search(r"[.!?][”\"'）)]*$", t))

    # heading from markup
    if block_kind == "heading":
        return ("fragment", "heading")

    # structural fragment shapes (before sentencehood)
    if _LIST_ITEM_END.search(t):
        # "the right to access;" / "...consent; and" / "...complain or"
        return ("fragment", "list_item")
    if t.endswith(":"):
        return ("fragment", "lead_in")

    # --- heading detection runs BEFORE sentencehood -----------------------
    # Short all-caps strings are section titles even when they contain a verb:
    # "SHARING OF PERSONAL DATA" and "HOW WE COLLECT AND USE INFORMATION" are
    # headings, not sentences. (The verb probe treats gerunds like "sharing"
    # and imperatives like "contact" as verbs, so we deliberately do NOT exempt
    # verb-bearing all-caps text -- a rare all-caps imperative misread as a
    # heading fragment is far less harmful than flooding the sentence view.)
    if _NUMBERED_HEADING.match(t) or _ROMAN_HEADING.match(t):
        return ("fragment", "heading")
    if _ALLCAPS_SHORT.match(t) and wc <= 10:
        return ("fragment", "heading")
    # short capitalized title with terminal punctuation (e.g. "Personal Data.")
    if has_terminal and wc <= 4 and not has_verb and t[0].isupper():
        return ("fragment", "heading")
    if wc <= 3 and not has_terminal:
        return ("fragment", "short")

    # --- subordinate-clause guard -----------------------------------------
    # A lone subordinate clause is not a complete thought, even when it ends
    # in a URL ("When you visit https://x.com"). A comma signals the main
    # clause follows ("If you have questions, please contact us").
    if _SUBORDINATE_START.match(t) and "," not in t:
        return ("fragment", "other")

    # --- URL/email-final complete thought ----------------------------------
    # A sentence-final URL/email is an independent complete-thought signal that
    # the verb probe may miss ("please visit ...", "please see ..."). Editors
    # omit a glued-on period after a URL (it would alter the resource locator):
    #   DB_1 convention: "...at example.com/terms"      (no period)
    #   DB_2 convention: "...at example.com/terms ."    (space, then period)
    # See _ends_in_url_email. This must run BEFORE the generic no-verb heading
    # heuristic below, or verbless URL-final sentences get lost as "heading".
    if wc >= 4 and _ends_in_url_email(t):
        return ("sentence", None)

    # short title-case titles without a verb and without punctuation
    # (e.g. "Effective Date") -- not URL-final (handled above)
    if (not has_terminal and not has_verb
            and not _has_finite_verb_signal(t) and wc <= 8):
        return ("fragment", "heading")

    # --- complete-thought sentencehood ------------------------------------
    # VERB_PROBE is a curated wordlist with inherent gaps ("pledges",
    # "conduct", "recommend", ...). The structural finite-verb signal catches
    # those without growing the wordlist: "we MAY need", "WE recommend".
    if wc >= 4 and (has_verb or _has_finite_verb_signal(t)):
        return ("sentence", None)

    return ("fragment", "other")


def extract_units(html: str) -> list[dict]:
    """End-to-end: HTML -> ordered list of units in document order.

    Each unit carries {unit_kind, fragment_type, text, block_kind, block_seq},
    where block_seq is the index of the containing block (paragraph/heading/
    list) and block_kind its type -- so the original block structure of the
    document can be reconstructed for a faithful "print to pdf" rendering.
    """
    units: list[dict] = []
    for block_seq, block in enumerate(extract_blocks(html)):
        for sent in split_sentences(block["text"]):
            kind, ftype = classify_unit(sent, block["kind"])
            units.append({
                "unit_kind": kind,
                "fragment_type": ftype,
                "text": sent,
                "block_kind": block["kind"],
                "block_seq": block_seq,
            })
    return units


# ---------------------------------------------------------------------------
# 4. Deterministic annotation alignment (ported from the parser's 3-tier logic)
# ---------------------------------------------------------------------------

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\xa0", " ").replace("\t", " ")
    for orig, repl in {"“": '"', "”": '"', "‘": "'", "’": "'", "`": "'",
                       "–": "-", "—": "-"}.items():
        text = text.replace(orig, repl)
    return re.sub(r"\s+", " ", text).strip().lower()


def load_annotations(csv_path: str) -> list[dict]:
    annotations = []
    if not os.path.exists(csv_path):
        return annotations
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row_idx, row in enumerate(reader):
            text_orig = row.get("Text", "")
            label = row.get("Label", "").strip()
            if not text_orig or not label:
                continue
            norm = normalize_text(text_orig)
            if not norm:
                continue
            annotations.append({
                "src_row": row_idx,
                "ranumb": row.get("RANumb", "").strip(),
                "text": text_orig,
                "normalized": norm,
                "label": label,
                "source_csv": csv_path,
            })
    return annotations


def align_annotations(units: list[dict], annotations: list[dict]):
    """Returns (aligned, unmatched, ambiguous).

    aligned:   list of (annotation_dict, unit_id, alignment_type)
    unmatched: list of annotation dicts
    ambiguous: list of (annotation_dict, [unit_id, ...])
    """
    unit_norms = [normalize_text(u["text"]) for u in units]
    unit_tokens = [set(n.split()) for n in unit_norms]

    aligned: list[tuple] = []
    unmatched = []
    ambiguous = []

    for ann in annotations:
        ann_norm = ann["normalized"]
        ann_tokens = set(ann_norm.split())

        # Case A: sub-sentence fragment contained in a single unit
        case_a = [i for i, n in enumerate(unit_norms) if n and ann_norm in n]
        if len(case_a) == 1:
            aligned.append((ann, units[case_a[0]]["id"], "annotation_in_sentence"))
            continue
        if len(case_a) > 1:
            ambiguous.append((ann, [units[i]["id"] for i in case_a]))
            continue

        # Case B: whole unit(s) contained inside the annotation paragraph span
        case_b = [i for i, n in enumerate(unit_norms)
                  if n and len(n) > 10 and n in ann_norm]
        if case_b:
            for i in case_b:
                aligned.append((ann, units[i]["id"], "sentence_in_annotation_paragraph"))
            continue

        # Case C: partial token overlap across a sentence boundary
        if len(ann_tokens) >= 3:
            best_i, best_overlap = None, 0.0
            for i, s_tokens in enumerate(unit_tokens):
                if not s_tokens:
                    continue
                inter = ann_tokens.intersection(s_tokens)
                if inter:
                    ov = len(inter) / len(ann_tokens)
                    if ov > best_overlap and ov >= 0.55:
                        best_overlap, best_i = ov, i
            if best_i is not None:
                aligned.append((ann, units[best_i]["id"], "partial_overlap"))
                continue

        unmatched.append(ann)

    return aligned, unmatched, ambiguous

In [ ]:
%%writefile backend/ingest.py
#!/usr/bin/env python3
"""Builds the normalized SQLite database for the C3PA Explorer.

Source layer: documents + annotations, verbatim from the C3PA dataset repo
(cloned into the submodule on first run).
Derived layer: text units re-extracted deterministically (backend/extract.py) and
annotations re-aligned to them, so every derived label is auditable.

Usage:
    python -m backend.ingest [--dataset-dir DIR] [--db PATH] [--max-docs N]
"""

import argparse
import csv
import glob
import os

from backend.db import connect, init_db
from backend.extract import align_annotations, extract_title, extract_units, load_annotations

APP_DIR = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(APP_DIR)
DEFAULT_DATASET = os.path.join(ROOT, "c3pa-sentence-label-parser", "C3PA_Dataset")
C3PA_REPO = "https://github.com/MaazBinMusa/C3PA_Dataset.git"


def ensure_dataset(dataset_dir: str) -> str:
    if not os.path.exists(os.path.join(dataset_dir, "Htmls")):
        if not os.path.exists(dataset_dir):
            os.makedirs(dataset_dir, exist_ok=True)
        print(f"[ingest] C3PA dataset not found; cloning into {dataset_dir} ...")
        os.system(f"git clone --depth 1 {C3PA_REPO} {dataset_dir}")
    return dataset_dir


def load_crawl_meta(dataset_dir: str, subset: str) -> dict[int, dict]:
    """Loads Crawl metadata for a subset. The repo ships one file per subset
    ('Crawl/db.csv', 'Crawl/ws.csv'); row i (after header) == doc number i+1."""
    meta = {}
    for path in glob.glob(os.path.join(dataset_dir, "Crawl", f"*{subset.lower()}*.csv")) \
            or glob.glob(os.path.join(dataset_dir, "Crawl", f"*{subset}*.csv")):
        with open(path, newline="", encoding="utf-8", errors="replace") as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader):
                meta[i + 1] = {
                    "link": (row.get("Link") or "").strip(),
                    "is_homepage": (row.get("IsHomepage") or "").strip(),
                    "textmatch_p": (row.get("Textmatch_P") or "").strip(),
                    "textmatch_s": (row.get("Textmatch_S") or "").strip(),
                    "textmatch_pp": (row.get("Textmatch_PP") or "").strip(),
                    "link_match": (row.get("Link_Match") or "").strip(),
                    "crawl_path": path,
                }
    return meta


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dataset-dir", default=DEFAULT_DATASET)
    ap.add_argument("--db", default=os.path.join(APP_DIR, "..", "data", "explorer.db"))
    ap.add_argument("--max-docs", type=int, default=None,
                    help="Limit to first N documents (for quick POC builds)")
    args = ap.parse_args()

    dataset_dir = os.path.abspath(args.dataset_dir)
    db_path = os.path.abspath(args.db)
    ensure_dataset(dataset_dir)

    import backend.db as dbmod

    dbmod.DB_PATH = db_path
    os.makedirs(os.path.dirname(db_path), exist_ok=True)
    conn = connect()
    cur = conn.cursor()
    cur.execute("PRAGMA foreign_keys = OFF")
    # drop app tables first (schema may have changed between builds)
    for t in ("alignment", "unit_labels", "text_units", "annotations", "documents",
              "sentence_labels", "sentences", "annotations_unmatched", "annotations_ambiguous"):
        cur.execute(f"DROP TABLE IF EXISTS {t}")
    cur.execute("PRAGMA foreign_keys = ON")
    init_db(conn)

    stats = {"docs": 0, "units": 0, "sentences": 0, "fragments": 0,
             "annotations": 0, "aligned": 0, "unmatched": 0, "ambiguous": 0,
             "single": 0, "multi": 0, "unlabeled": 0, "high_conf": 0}

    doc_pairs = []
    for subset in ("DB", "WS"):
        ann_dir = os.path.join(dataset_dir, "Annotations", subset)
        html_dir = os.path.join(dataset_dir, "Htmls", subset)
        if not (os.path.isdir(ann_dir) and os.path.isdir(html_dir)):
            continue
        crawl = load_crawl_meta(dataset_dir, subset)
        for csv_path in sorted(glob.glob(os.path.join(ann_dir, "*.csv"))):
            num = os.path.splitext(os.path.basename(csv_path))[0]
            html_path = os.path.join(html_dir, f"{num}.html")
            if not os.path.exists(html_path):
                continue
            doc_pairs.append((f"{subset}_{num}", subset, int(num), html_path, csv_path, crawl))

    if args.max_docs:
        doc_pairs = doc_pairs[: args.max_docs]
    print(f"[ingest] processing {len(doc_pairs)} documents ...")

    for doc_id, subset, num, html_path, csv_path, crawl in doc_pairs:
        crawl_meta = crawl.get(num, {})
        # derived: extract units deterministically
        with open(html_path, encoding="utf-8", errors="replace") as f:
            html = f.read()
        cur.execute(
            """INSERT INTO documents (doc_id, subset, num, title, link, is_homepage,
               textmatch_p, textmatch_s, textmatch_pp, link_match, html_path, crawl_path)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
            (doc_id, subset, num, extract_title(html),
             crawl_meta.get("link"), crawl_meta.get("is_homepage"),
             crawl_meta.get("textmatch_p"), crawl_meta.get("textmatch_s"),
             crawl_meta.get("textmatch_pp"), crawl_meta.get("link_match"),
             html_path, crawl_meta.get("crawl_path")),
        )

        raw_units = extract_units(html)
        units = []
        for i, u in enumerate(raw_units, start=1):
            units.append({**u, "id": f"{doc_id}_U{i}", "doc_id": doc_id, "position": i})
            cur.execute(
                """INSERT INTO text_units (unit_id, doc_id, position, block_seq, block_kind,
                   unit_text, unit_kind, fragment_type, label_category, annotator_count,
                   source_annotation_count, alignment_types)
                   VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
                (units[-1]["id"], doc_id, i, u["block_seq"], u["block_kind"],
                 u["text"], u["unit_kind"], u["fragment_type"],
                 # initial category: sentences start 'unlabeled' (measurement-
                 # relative -- no annotation signal yet); fragments are
                 # 'not-eligible' by design (see note in backend/db.py)
                 "unlabeled" if u["unit_kind"] == "sentence" else "not-eligible",
                 0, 0, None),
            )
            stats["units"] += 1
            stats["sentences" if u["unit_kind"] == "sentence" else "fragments"] += 1

        # source: annotations (verbatim)
        annotations = load_annotations(csv_path)
        ann_id_by_src = {}
        for ann in annotations:
            cur.execute(
                """INSERT INTO annotations (doc_id, ranumb, text, label, status, src_row, source_csv)
                   VALUES (?,?,?,?,?,?,?)""",
                (doc_id, ann["ranumb"], ann["text"], ann["label"], "aligned",
                 ann["src_row"], ann["source_csv"]),
            )
            ann_id_by_src[ann["src_row"]] = cur.lastrowid
            stats["annotations"] += 1

        # derived: align annotations to units (3-tier, deterministic)
        aligned, unmatched, ambiguous = align_annotations(units, annotations)

        for ann in unmatched:
            cur.execute(
                "UPDATE annotations SET status='unmatched' WHERE doc_id=? AND src_row=?",
                (doc_id, ann["src_row"]),
            )
            stats["unmatched"] += 1
        for ann, unit_ids in ambiguous:
            cur.execute(
                "UPDATE annotations SET status='ambiguous', matched_units=? WHERE doc_id=? AND src_row=?",
                (";".join(unit_ids), doc_id, ann["src_row"]),
            )
            stats["ambiguous"] += 1

        # Alignment provenance is recorded for EVERY unit (sentences and
        # fragments alike) so the audit trail is complete: you can always see
        # where an annotation landed. But label categories are applied to
        # SENTENCES ONLY. The project's goal is reliable sentence/label pairs;
        # fragments are by definition not sentences, so a label on one is noise
        # for that purpose (see the note in backend/db.py). Fragment units keep the
        # distinct category 'not-eligible' with no unit_labels rows; their
        # alignments remain. 'unlabeled' sentences are measurement-relative:
        # no annotation signal for this version -- NOT label-inapplicable.
        evidence: dict[str, dict] = {}
        unit_kind = {u["id"]: u["unit_kind"] for u in units}
        for ann, unit_id, align_type in aligned:
            cur.execute(
                "INSERT INTO alignment (unit_id, annotation_id, alignment_type) VALUES (?,?,?)",
                (unit_id, ann_id_by_src[ann["src_row"]], align_type),
            )
            stats["aligned"] += 1
            if unit_kind[unit_id] != "sentence":
                continue
            ev = evidence.setdefault(unit_id, {"labels": set(), "annotators": set(),
                                               "align": set(), "count": 0})
            ev["labels"].add(ann["label"])
            ev["annotators"].add(ann["ranumb"])
            ev["align"].add(align_type)
            ev["count"] += 1

        for unit in units:
            if unit["unit_kind"] != "sentence":
                continue
            ev = evidence.get(unit["id"])
            if not ev:
                continue
            labels = sorted(ev["labels"])
            annotators = sorted(ev["annotators"])
            category = ("unlabeled" if not labels
                        else "single_label" if len(labels) == 1 else "multi_label")
            cur.execute(
                """UPDATE text_units SET label_category=?, annotator_count=?,
                   source_annotation_count=?, alignment_types=? WHERE unit_id=?""",
                (category, len(annotators), ev["count"], ";".join(sorted(ev["align"])),
                 unit["id"]),
            )
            for lbl in labels:
                cur.execute("INSERT INTO unit_labels (unit_id, label) VALUES (?,?)",
                            (unit["id"], lbl))
            stats[{"single_label": "single", "multi_label": "multi",
                   "unlabeled": "unlabeled"}[category]] += 1
            if category == "single_label" and len(annotators) >= 2:
                stats["high_conf"] += 1

        stats["docs"] += 1
        if stats["docs"] % 50 == 0:
            print(f"  ...{stats['docs']} docs done")

    # all fragments are 'not-eligible' by design; all remaining sentences are
    # 'unlabeled' (no annotation signal for this version)
    stats["unlabeled"] = stats["sentences"] - stats["single"] - stats["multi"]

    conn.commit()
    conn.close()

    print("\n" + "=" * 60)
    print(" C3PA EXPLORER DB BUILD SUMMARY")
    print("=" * 60)
    for k, v in stats.items():
        print(f"  {k:<12} {v:>9,}")
    print("=" * 60)
    print(f"DB written to {db_path}")


if __name__ == "__main__":
    main()

In [ ]:
# 3. Only build-time dep: beautifulsoup4, pinned (matches the recorded baseline;
# unpinned bs4 could drift counts). It pulls soupsieve + typing-extensions.
# Extraction uses stdlib html.parser only -- no lxml/html5lib, spacy, or fastapi.
!pip install -q beautifulsoup4==4.15.0
import bs4
print("bs4", bs4.__version__)


In [ ]:
# 4. Build the snapshot DB with the UNCHANGED pipeline.
#
# Full corpus by default. For a QUICK SMOKE BUILD set MAX_DOCS, e.g.:
#   MAX_DOCS = 5
# When set, --max-docs N is passed to ingest and the manifest is flagged
# partial (counts are then for the partial slice, not the full corpus).
MAX_DOCS = None
PARTIAL = MAX_DOCS is not None

import os
import subprocess
import sys

cmd = [
    sys.executable, "-m", "backend.ingest",
    "--dataset-dir", "C3PA_Dataset",
    "--db", "explorer.db",
]
if PARTIAL:
    cmd += ["--max-docs", str(int(MAX_DOCS))]
print("+", " ".join(cmd))
subprocess.check_call(cmd)
print("ingest exit: OK")


In [ ]:
# 5. Embed the manifest table into explorer.db
#
# Counts use the SAME SQL + key names as backend/services/stats_service.py
# get_stats_data, so manifest counts can never drift from /api/stats.
# db_sha256 is hashed LAST, over the closed final file (see below).

import hashlib
import json
import os
import sqlite3
from datetime import datetime, timezone

# Agreed snapshot schema version -- must equal browser/js/core/snapshot-version.js.
SNAPSHOT_SCHEMA_VERSION = "c3pa-explorer-snapshot-v1"
# Baked at GENERATION time by scripts/build_colab_notebook.py (C3PA_EXPLORER_REV
# env override, else repo git HEAD, else "vendored-cells" + timestamp).
code_rev = "e33e94d90b2f3fa5ef8590267d3bf77e202ae1cc"

DB_PATH = "explorer.db"
MAX_DOCS = globals().get("MAX_DOCS")              # defined by the ingest cell; survives re-runs
PARTIAL = MAX_DOCS is not None
dataset_commit = globals().get("dataset_commit") or "unknown"


def cnt(conn, sql, params=()):
    return conn.execute(sql, params).fetchone()[0]


conn = sqlite3.connect(DB_PATH)                   # read-write: the manifest table is new
conn.row_factory = sqlite3.Row
try:
    # --- counts: same SQL + key names as stats_service.get_stats_data ---
    kinds = {r["unit_kind"]: r["n"] for r in conn.execute(
        "SELECT unit_kind, COUNT(*) n FROM text_units GROUP BY unit_kind")}
    cats = {r["label_category"]: r["n"] for r in conn.execute(
        "SELECT label_category, COUNT(*) n FROM text_units WHERE unit_kind='sentence' GROUP BY label_category")}
    counts = {
        "documents": cnt(conn, "SELECT COUNT(*) FROM documents"),
        "units": cnt(conn, "SELECT COUNT(*) FROM text_units"),
        "sentences": kinds.get("sentence", 0),
        "fragments": kinds.get("fragment", 0),
        "annotations": cnt(conn, "SELECT COUNT(*) FROM annotations"),
        "aligned": cnt(conn, "SELECT COUNT(*) FROM annotations WHERE status='aligned'"),
        "unmatched": cnt(conn, "SELECT COUNT(*) FROM annotations WHERE status='unmatched'"),
        "ambiguous": cnt(conn, "SELECT COUNT(*) FROM annotations WHERE status='ambiguous'"),
        "single_label_sentences": cats.get("single_label", 0),
        "multi_label_sentences": cats.get("multi_label", 0),
        "unlabeled_sentences": cats.get("unlabeled", 0),
        "high_confidence_single": cnt(
            conn, "SELECT COUNT(*) FROM text_units WHERE label_category='single_label' AND annotator_count>=2"),
        "subsets": {r["subset"]: r["n"] for r in
                    conn.execute("SELECT subset, COUNT(*) n FROM documents GROUP BY subset")},
    }

    expected_docs = 400 if not PARTIAL else 1
    assert counts["documents"] >= expected_docs, (
        f"documents={counts['documents']} below expected {expected_docs}; dataset incomplete?"
    )
    if PARTIAL:
        counts["partial"] = True
        counts["max_docs"] = int(MAX_DOCS)

    conn.execute("CREATE TABLE IF NOT EXISTS manifest (key TEXT PRIMARY KEY, value TEXT)")
    conn.executemany("INSERT OR REPLACE INTO manifest (key, value) VALUES (?,?)", [
        ("schema_version", SNAPSHOT_SCHEMA_VERSION),
        ("dataset_commit", dataset_commit),
        ("code_rev", code_rev),
        ("counts", json.dumps(counts, sort_keys=True, separators=(",", ":"))),
        ("built_at", datetime.now(timezone.utc).isoformat()),
    ])
    conn.commit()
    conn.execute("VACUUM")
finally:
    conn.close()

for side in ("-wal", "-shm"):
    assert not os.path.exists(DB_PATH + side),         f"stray {DB_PATH}{side} left behind; checkpoint/close failed"

# db_sha256 LAST, over the CLOSED final file. The stored value covers the
# delivered bytes with the checksum row itself excluded (a file cannot contain
# the hash of its own final bytes). Transport integrity is `PRAGMA
# integrity_check` + schema_version; db_sha256 is the provenance checksum.
db_sha256 = hashlib.sha256(open(DB_PATH, "rb").read()).hexdigest()

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany("INSERT OR REPLACE INTO manifest (key, value) VALUES (?,?)", [
        ("db_sha256", db_sha256),
    ])
    conn.commit()
    conn.execute("VACUUM")
finally:
    conn.close()

for side in ("-wal", "-shm"):
    assert not os.path.exists(DB_PATH + side),         f"stray {DB_PATH}{side} left behind; checkpoint/close failed"

print("=== manifest ===")
conn = sqlite3.connect(DB_PATH)
try:
    for k, v in sorted(conn.execute("SELECT key, value FROM manifest")):
        shown = v if len(v) <= 120 else v[:120] + "..."
        print(f"  {k:<14} {shown}")
finally:
    conn.close()
print("db size:", os.path.getsize(DB_PATH), "bytes")


In [ ]:
# 6. Download the snapshot + optional Google Drive copy.
#
# files.download streams the whole file through kernel comms into a browser
# Blob (~153 MB) -- fine on a desktop browser; the Drive copy is the robust
# path for large files. This cell never mounts Drive (it blocks on an
# interactive auth prompt); an already-mounted Drive is picked up if present.
import os
from google.colab import files

files.download("explorer.db")

MOUNT_DRIVE = False   # set True to also copy to MyDrive/c3pa_explorer/
if MOUNT_DRIVE and os.path.exists("/content/drive/MyDrive"):
    import shutil
    dest_dir = "/content/drive/MyDrive/c3pa_explorer"
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, "explorer.db")
    shutil.copy("explorer.db", dest)
    print("copied to:", dest)
else:
    print("Drive copy skipped (MOUNT_DRIVE=False or /content/drive/MyDrive not mounted)")
